# LIBRERIAS


In [1]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# EXPLORACION DE DATOS

Plan de trabajo con los datos:
Dado que lo que se quiere crear es proceso para la gestión de insumos, inicialmente analizaremos en detalle las fuentes entregadas por el cliente ISSETA HNOS.
[Cargar en Python y hacer una exploración descriptiva]


Para comenzar el análisis nos entregaron tres archivos en formato excel:
1.	Productos ISSETTA.xlsx: catálogo de productos, con tres campos.
	Productos: tipo texto
	Categoría: texto (con faltante de datos)
	IVA: float (también con faltante de datos)

2.	Planilla carga materiales.xlsx: contiene esencialmente el stock, información descriptiva de los materiales y el almacén donde se guardan, con los siguientes campos	
Campo	Tipo	Detalle
codigo	Texto	Id Material
descripcion	Texto	ATRIBUTOS descriptivos del material
categoria	Texto	
sub-categoria	Texto	
unidad-medida	Texto	
MATERIAL	Texto	
caracteristica2	Texto	
caracteristica3	Texto	
caracteristica4	Texto	
precio_actual (u$s)	float	
almacen	texto	Lugar de almacenamiento
cantidad	int	Stock (dato variable)
3.	F-PROD-01-B registro produccion prueba.xlsx: registro de tareas realizadas por los empleados, por proyecto con cuantificación de tareas, fecha de realización y duración del proceso. Cuenta con múltiples solapas, una por cada empleado. Deberían ser unificada para poder ser tratadas como un dataset, tomando al empleado como un atributo más de cada registro. 


## 	Productos ISSETTA.xlsx

In [62]:
# Cargamos el dataset Productos ISSETTA.xslx
productos = pd.read_excel('Productos ISSETTA.xlsx')    

In [63]:
productos.head()

,Productos,Categoria,IVA
0,ABRAZADERA DE 8 A 22MM 304,General,0.21
1,"ABRAZADERA RAPIDA P/PARED 7/8"" P/CAÑO DAISA",General,0.21
2,Abrazaderas acero inoxidable - chicas 10-16mm,General,0.21
3,ABRAZADERAS INOX,General,0.21
4,ABS GRILON3 1KG BLANCO,General,0.21


In [65]:
productos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Productos  1091 non-null   object 
 1   Categoria  237 non-null    object 
 2   IVA        234 non-null    float64
dtypes: float64(1), object(2)
memory usage: 26.8+ KB


In [66]:
# Cuantas categorias de producto hay y cuantos productos hay por categoria
productos['Categoria'].value_counts()

General        212
Electronica     25
Name: Categoria, dtype: int64

## Planilla Carga Materiales.xlsx

In [83]:
# Cargamos el dataset que creemos que aporta más información, llamado Planilla Carga Materiales.xslx, la primer línea no contiene infomración relevante, por lo que la ignoramos
stock = pd.read_excel('Planilla Carga Materiales.xlsx', skiprows=1)


In [84]:
stock.head()

,codigo,descripcion,categoria,sub-categoria,unidad-medida,MATERIAL,caracteristica2,caracteristica3,caracteristica4,precio_actual (u$s),almacen,cantidad,Unnamed: 12
0,NaN,ACERO COMUN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
1,51114.0,Ruleman,mecanica,RULEMANES,unidad,acero comun,NaN,NaN,NaN,NaN,1.0,0.0,NaN
2,UC505,Ruleman,mecanica,RULEMANES,unidad,acero comun,NaN,NaN,NaN,NaN,NaN,0.0,NaN
3,51205.0,Ruleman,mecanica,RULEMANES,unidad,acero comun,NaN,NaN,NaN,NaN,1.0,0.0,NaN
4,0605AC,Ruleman,mecanica,RULEMANES,unidad,acero comun,NaN,NaN,NaN,NaN,1.0,8.0,NaN


In [85]:
stock.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   codigo               365 non-null    object 
 1   descripcion          492 non-null    object 
 2   categoria            382 non-null    object 
 3   sub-categoria        397 non-null    object 
 4   unidad-medida        478 non-null    object 
 5   MATERIAL             417 non-null    object 
 6   caracteristica2      234 non-null    object 
 7   caracteristica3      3 non-null      object 
 8   caracteristica4      0 non-null      float64
 9   precio_actual (u$s)  19 non-null     object 
 10  almacen              461 non-null    object 
 11  cantidad             478 non-null    object 
 12  Unnamed: 12          9 non-null      object 
dtypes: float64(1), object(12)
memory usage: 51.1+ KB


Se observan muchos problemas fundamentales en esta tabla:
1- Hay más descripciones que códigos, esto significa que el campo código no representa clave primaria para esta tabla, y por lo tanto no identifica unívocamente todos los materiales. Hay muchos campos código en estado null
2- Se observa en la fuente que los elementos se encuentran visualmente agrupados (en subcategorias), dicha agrupación no puede ser registrada correctamente en esta primer lectura. Proponemos hacer un ajuste a la fuente.
3- Proponemos hacer dos ajustes a la fuente:
    a- Agregar un campo con la categoría faltante
    b- Crear un código unico basado en varios campos de cada registro


Grabaremos la fuente con otro nombre para no modificar la original. El campo caracterisitca4 será cambiado su nombre por grupo y allí se cargará con una formula de excel el nuevo dato.

In [18]:
# El nuevo dataset se llama planilla_carga_mat_corr.xlsx, lo cargamos y lo exploramos, sigue teniendo la primer línea sin información relevante, por lo que la ignoramos
stock_corr = pd.read_excel('planilla_carga_mat_corr.xlsx', skiprows=1)

In [19]:
stock_corr.head()

,id,codigo,descripcion,categoria,sub-categoria,unidad-medida,MATERIAL,caracteristica2,caracteristica3,grupo,precio_actual (u$s),almacen,cantidad,datos,grup_
0,-ACERO COMUN,NaN,ACERO COMUN,NaN,NaN,NaN,NaN,NaN,NaN,ACERO COMUN,NaN,1,NaN,NaN,ACERO COMUN
1,51114-Ruleman,51114,Ruleman,mecanica,RULEMANES,unidad,acero comun,NaN,NaN,ACERO COMUN,NaN,1,0,NaN,NaN
2,UC505-Ruleman,UC505,Ruleman,mecanica,RULEMANES,unidad,acero comun,NaN,NaN,ACERO COMUN,NaN,NaN,0,NaN,NaN
3,51205-Ruleman,51205,Ruleman,mecanica,RULEMANES,unidad,acero comun,NaN,NaN,ACERO COMUN,NaN,1,0,NaN,NaN
4,0605AC-Ruleman,0605AC,Ruleman,mecanica,RULEMANES,unidad,acero comun,NaN,NaN,ACERO COMUN,NaN,1,8,NaN,NaN


In [20]:
stock_corr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   502 non-null    object
 1   codigo               361 non-null    object
 2   descripcion          497 non-null    object
 3   categoria            382 non-null    object
 4   sub-categoria        397 non-null    object
 5   unidad-medida        478 non-null    object
 6   MATERIAL             417 non-null    object
 7   caracteristica2      234 non-null    object
 8   caracteristica3      3 non-null      object
 9   grupo                502 non-null    object
 10  precio_actual (u$s)  19 non-null     object
 11  almacen              461 non-null    object
 12  cantidad             478 non-null    object
 13  datos                9 non-null      object
 14  grup_                8 non-null      object
dtypes: object(15)
memory usage: 59.0+ KB


In [21]:
# Queremos saber cuantos grupos de materiales hay, para eso contamos los valores únicos de la columna 'grupo'
stock_corr['grupo'].nunique()

9

In [22]:
# Elementos por grupo
stock_corr['grupo'].value_counts()

ACERO COMUN        209
MENINO             107
INOXIDABLE          75
MECHAS              72
VARIOS              15
ELECTRONICA          8
DISCOS DE CORTE      8
0                    5
PINTURAS             3
Name: grupo, dtype: int64

Se observa que este data set que maneja stocks, maneja materiales de uso e insumos indistintamente

In [23]:
# Elementos por descripcion
stock_corr['descripcion'].value_counts()

MECHA                        48
Ruleman                      42
RETENES                      39
ALLEN INOX                   24
RODAMIENTO                   21
                             ..
LSP-1                         1
DMS 42E                       1
VENTOSA                       1
MS 4-WP                       1
RODILLO PARA LIJADO INOX      1
Name: descripcion, Length: 170, dtype: int64

In [24]:
# Elementos por categoria
stock_corr['categoria'].value_counts()

mecanica                  372
electronica                 7
MEcanica                    1
FUENTE                      1
NEBL-M8G4-E-0,3-N-M8G4      1
Name: categoria, dtype: int64

Cantidad y precio deberían ser datos numéricos

In [26]:
# necesitamos convertir la columna cantidad y precio actual a numérico, primero vemos que tipo de datos tienen no numericos para poder tomar la descicion de como convertirlos
stock_corr['cantidad'].unique()


array([nan, 0, 8, 3, 23, 13, 6, 14, 2, 1, 5, 9, 4, 21, 29, 44, 35, 20, 28,
       332, 90, 708, 67, 36, '10 KG', '13,52 KG', 200, 503, 110, 95, 52,
       360, -4, -6, 102, 154, 48, 115, 259, 16, 7, 24, 10, 121, 22, 26,
       196, 148, 185, 199, 417, 138, 40, 32, 79, 580, -7, 12, -14, 31, 15,
       -1, 27, 69, 241, 256, 92, 33, 335, 45, 100, 158, 236, 232, 39, 202,
       212, 292, 240, 216, '/', -22, 77, 394, 11, -15, 58, 17, '--83', 41,
       50, '50CM', '10 MTR', datetime.datetime(2025, 6, 4, 0, 0),
       datetime.datetime(2025, 3, 9, 0, 0), -44, -200, 25, 131, 435],
      dtype=object)

Llama la atención las cantidades negativas, y otro tipo de datos que no se condicen con un stock.

In [27]:
# veamos que pasa con los datos precio_actual que también es un dato numérico pero tiene el mismo problema que cantidad
stock_corr['precio_actual (u$s)'].unique()

array([nan, 'USD 638', 'USD 6,38', 'USD 010', 'USD 0,80', 'USD 0,08',
       'USD 0,26', 2.33, 0.44, 0.05, 0.11, 0.41, 0.25, 0.29, 18, 0.12,
       0.07, 0.46, '5,73 C/U'], dtype=object)

Entendemos que son todos valores unitarios, y están todos en dólares. Entiendo que se puede hacer una limpieza de los datos que no son números de estos campos

## F-PROD-01-B registro produccion prueba.xlsx

In [2]:
# Debemos estructurar el excel F-PROD-01-B registro produccion prueba.xlsx para poder cargarlo y analizarlo, lo cargamos y exploramos. La información relevante comienza a partir de la fila 5
# Está organizada en 9 hojas, cada una con información de un empleado diferente, cada hoja tiene estructura similar, y el nombre de la hoja es el nombre del empleado,
#  lo que nos puede servir para identificar a cada empleado en el análisis, que deberá agregarse como un atributo al dataset
# Debemos armar un dataset unificado con la información de las 9 hojas, para eso utilizamos el método pd.concat() para concatenar los dataframes de cada hoja, y agregamos una columna 'empleado' para identificar a cada empleado
produccion = pd.DataFrame()
for sheet in pd.ExcelFile('F-PROD-01-B registro produccion prueba.xlsx').sheet_names:
    df = pd.read_excel('F-PROD-01-B registro produccion prueba.xlsx', sheet_name=sheet, skiprows=4)
    df['empleado'] = sheet
    produccion = pd.concat([produccion, df], ignore_index=True)
produccion.head()



,Unnamed: 0,FECHA,IDENTIFICACION DE PROYECTO,IDENTIFICACION DE PLANO,OPERACIÓN,DESCRIPCIÓN OPERACIÓN,Unnamed: 6,CANTIDAD (UNIDADES POR PLANO),TIEMPO DE EJECUCIÓN REAL,Unnamed: 9,Unnamed: 10,TAKE TIME,Unnamed: 12,Unnamed: 13,OBSERVACION,LIBERACION PIEZAS (LIDER),Unnamed: 16,Unnamed: 17,empleado
0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,HORA Inicio,HORA Final,Tiempo de paradas,Minutos totales de produccion,Unidades,resultado Min/U,NaN,NaN,NaN,NaN,ALARCON
1,NaN,2026-03-17,G800,P01 03 009,FRESADO,CORREDERA EN 4 PERFORACIONES,NaN,1.0,07:10:00,09:43:00,10,143.0,1,143.0,NaN,NaN,NaN,NaN,ALARCON
2,NaN,NaT,G800,P01 06 001,FRESADO,CORREDERA EN 8 PERFORACIONES,NaN,1.0,09:50:00,10:05:00,2,13.0,1,13.0,NaN,NaN,NaN,NaN,ALARCON
3,NaN,NaT,G800,SIN DATO,ARMADORA,REFORMA PALLETIZADORA,NaN,1.0,10:07:00,11:08:00,11,50.0,1,50.0,NaN,NaN,NaN,NaN,ALARCON
4,NaN,NaT,BAÑADORA EMPANADAS,SIN PLANO,CORTE CHAPA,CORTAR TAPA SOPLADOR INOX 200 x 110,NaN,2.0,11:10:00,11:25:00,2,13.0,2,6.5,EN TOTAL PARA EL TRABAJO DE LA BAÑADORA QUE FU...,NaN,NaN,NaN,ALARCON


In [35]:
produccion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573 entries, 0 to 572
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Unnamed: 0                     0 non-null      float64       
 1   FECHA                          42 non-null     datetime64[ns]
 2   IDENTIFICACION  DE PROYECTO    169 non-null    object        
 3   IDENTIFICACION DE PLANO        121 non-null    object        
 4   OPERACIÓN                      144 non-null    object        
 5   DESCRIPCIÓN OPERACIÓN          145 non-null    object        
 6   Unnamed: 6                     0 non-null      float64       
 7   CANTIDAD (UNIDADES POR PLANO)  143 non-null    float64       
 8   TIEMPO DE EJECUCIÓN REAL       573 non-null    object        
 9   Unnamed: 9                     573 non-null    object        
 10  Unnamed: 10                    573 non-null    object        
 11  TAKE TIME          

In [3]:
# empezamos a limpiar el dataste. Primero eliminamos la primer columna.
produccion = produccion.drop(produccion.columns[0], axis=1)

In [37]:
produccion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573 entries, 0 to 572
Data columns (total 18 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   FECHA                          42 non-null     datetime64[ns]
 1   IDENTIFICACION  DE PROYECTO    169 non-null    object        
 2   IDENTIFICACION DE PLANO        121 non-null    object        
 3   OPERACIÓN                      144 non-null    object        
 4   DESCRIPCIÓN OPERACIÓN          145 non-null    object        
 5   Unnamed: 6                     0 non-null      float64       
 6   CANTIDAD (UNIDADES POR PLANO)  143 non-null    float64       
 7   TIEMPO DE EJECUCIÓN REAL       573 non-null    object        
 8   Unnamed: 9                     573 non-null    object        
 9   Unnamed: 10                    573 non-null    object        
 10  TAKE TIME                      573 non-null    object        
 11  Unnamed: 12        

In [4]:
# Debemos quitar los registros que no contienen información en el campo Descripción Operacion, para eso utilizamos el método dropna() con el parámetro subset=['DESCRIPCIÓN OPERACIÓN']
produccion = produccion.dropna(subset=['DESCRIPCIÓN OPERACIÓN'])


In [39]:
produccion.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 145 entries, 1 to 546
Data columns (total 18 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   FECHA                          41 non-null     datetime64[ns]
 1   IDENTIFICACION  DE PROYECTO    123 non-null    object        
 2   IDENTIFICACION DE PLANO        121 non-null    object        
 3   OPERACIÓN                      143 non-null    object        
 4   DESCRIPCIÓN OPERACIÓN          145 non-null    object        
 5   Unnamed: 6                     0 non-null      float64       
 6   CANTIDAD (UNIDADES POR PLANO)  142 non-null    float64       
 7   TIEMPO DE EJECUCIÓN REAL       145 non-null    object        
 8   Unnamed: 9                     145 non-null    object        
 9   Unnamed: 10                    145 non-null    object        
 10  TAKE TIME                      145 non-null    object        
 11  Unnamed: 12        

In [70]:
produccion

,FECHA,IDENTIFICACION DE PROYECTO,IDENTIFICACION DE PLANO,OPERACIÓN,DESCRIPCIÓN OPERACIÓN,Unnamed: 6,CANTIDAD (UNIDADES POR PLANO),TIEMPO DE EJECUCIÓN REAL,Unnamed: 9,Unnamed: 10,TAKE TIME,Unnamed: 12,Unnamed: 13,OBSERVACION,LIBERACION PIEZAS (LIDER),Unnamed: 16,Unnamed: 17,empleado
1,2026-03-17,G800,P01 03 009,FRESADO,CORREDERA EN 4 PERFORACIONES,NaN,1.0,07:10:00,09:43:00,10,143.0,1,143.0,NaN,NaN,NaN,NaN,ALARCON
2,NaT,G800,P01 06 001,FRESADO,CORREDERA EN 8 PERFORACIONES,NaN,1.0,09:50:00,10:05:00,2,13.0,1,13.0,NaN,NaN,NaN,NaN,ALARCON
3,NaT,G800,SIN DATO,ARMADORA,REFORMA PALLETIZADORA,NaN,1.0,10:07:00,11:08:00,11,50.0,1,50.0,NaN,NaN,NaN,NaN,ALARCON
4,NaT,BAÑADORA EMPANADAS,SIN PLANO,CORTE CHAPA,CORTAR TAPA SOPLADOR INOX 200 x 110,NaN,2.0,11:10:00,11:25:00,2,13.0,2,6.5,EN TOTAL PARA EL TRABAJO DE LA BAÑADORA QUE FU...,NaN,NaN,NaN,ALARCON
5,NaT,BAÑADORA EMPANADAS,SIN PLANO,CORTE CHAPA,CORTAR TAPA SOPLADOR INOX 200 x 110,NaN,2.0,11:25:00,12:09:00,5,39,2,19.5,NaN,NaN,200.0,MINUTOS TOTALES,ALARCON
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
542,NaT,NO APLICA,NO APLICA,LIMPIEZA,LIMPIEZA PANTOGRAFO,NaN,1.0,15:30:00,16:00:00,0,30.0,1,30.0,NaN,NaN,NaN,NaN,VERA
543,2026-03-31,G800,NO APLICA,CORTE LASER,DISEÑO Y CORTE EN PANTOGRAFO DE PLACAS P SOPORTE,NaN,3.0,07:00:00,08:30:00,10,80,3,26.666667,NaN,NaN,NaN,NaN,VERA
544,NaT,G800,NO APLICA,CORTE LASER,ADAPTACION SOPORTE,NaN,2.0,08:30:00,10:30:00,0,120.0,2,60.0,NaN,NaN,NaN,NaN,VERA
545,NaT,G800,NO APLICA,MECANICA,"ROSCADO, PULIDO SOPORTE",NaN,2.0,10:30:00,11:30:00,0,60.0,1,60.0,NaN,NaN,NaN,NaN,VERA


In [5]:
# Debemos eliminar la columna que tiene todos sus valores nulos, para eso utilizamos el método drop() con el parámetro columns=['Unnamed: 6']
produccion = produccion.drop(columns=['Unnamed: 6'])

In [6]:
# Debemos completar las fechas que no tienen informacion, para eso utilizamos el método fillna() con el parámetro method='ffill' para rellenar las fechas hacia adelante
produccion['FECHA'] = produccion['FECHA'].fillna(method='ffill')

In [7]:
produccion.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 145 entries, 1 to 546
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   FECHA                          145 non-null    datetime64[ns]
 1   IDENTIFICACION  DE PROYECTO    123 non-null    object        
 2   IDENTIFICACION DE PLANO        121 non-null    object        
 3   OPERACIÓN                      143 non-null    object        
 4   DESCRIPCIÓN OPERACIÓN          145 non-null    object        
 5   CANTIDAD (UNIDADES POR PLANO)  142 non-null    float64       
 6   TIEMPO DE EJECUCIÓN REAL       145 non-null    object        
 7   Unnamed: 9                     145 non-null    object        
 8   Unnamed: 10                    145 non-null    object        
 9   TAKE TIME                      145 non-null    object        
 10  Unnamed: 12                    145 non-null    object        
 11  Unnamed: 13        

In [8]:
produccion

,FECHA,IDENTIFICACION DE PROYECTO,IDENTIFICACION DE PLANO,OPERACIÓN,DESCRIPCIÓN OPERACIÓN,CANTIDAD (UNIDADES POR PLANO),TIEMPO DE EJECUCIÓN REAL,Unnamed: 9,Unnamed: 10,TAKE TIME,Unnamed: 12,Unnamed: 13,OBSERVACION,LIBERACION PIEZAS (LIDER),Unnamed: 16,Unnamed: 17,empleado
1,2026-03-17,G800,P01 03 009,FRESADO,CORREDERA EN 4 PERFORACIONES,1.0,07:10:00,09:43:00,10,143.0,1,143.0,NaN,NaN,NaN,NaN,ALARCON
2,2026-03-17,G800,P01 06 001,FRESADO,CORREDERA EN 8 PERFORACIONES,1.0,09:50:00,10:05:00,2,13.0,1,13.0,NaN,NaN,NaN,NaN,ALARCON
3,2026-03-17,G800,SIN DATO,ARMADORA,REFORMA PALLETIZADORA,1.0,10:07:00,11:08:00,11,50.0,1,50.0,NaN,NaN,NaN,NaN,ALARCON
4,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,CORTE CHAPA,CORTAR TAPA SOPLADOR INOX 200 x 110,2.0,11:10:00,11:25:00,2,13.0,2,6.5,EN TOTAL PARA EL TRABAJO DE LA BAÑADORA QUE FU...,NaN,NaN,NaN,ALARCON
5,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,CORTE CHAPA,CORTAR TAPA SOPLADOR INOX 200 x 110,2.0,11:25:00,12:09:00,5,39,2,19.5,NaN,NaN,200.0,MINUTOS TOTALES,ALARCON
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
542,2026-03-30,NO APLICA,NO APLICA,LIMPIEZA,LIMPIEZA PANTOGRAFO,1.0,15:30:00,16:00:00,0,30.0,1,30.0,NaN,NaN,NaN,NaN,VERA
543,2026-03-31,G800,NO APLICA,CORTE LASER,DISEÑO Y CORTE EN PANTOGRAFO DE PLACAS P SOPORTE,3.0,07:00:00,08:30:00,10,80,3,26.666667,NaN,NaN,NaN,NaN,VERA
544,2026-03-31,G800,NO APLICA,CORTE LASER,ADAPTACION SOPORTE,2.0,08:30:00,10:30:00,0,120.0,2,60.0,NaN,NaN,NaN,NaN,VERA
545,2026-03-31,G800,NO APLICA,MECANICA,"ROSCADO, PULIDO SOPORTE",2.0,10:30:00,11:30:00,0,60.0,1,60.0,NaN,NaN,NaN,NaN,VERA


In [9]:
# Se observa en la fuente que el campo 'IDENTIFICACION DE PROYECTO' presenta el mismo problema que el campo FECHA, por lo que también debemos rellenar los valores faltantes hacia adelante utilizando el método fillna() con el parámetro method='ffill'
produccion['IDENTIFICACION  DE PROYECTO'] = produccion['IDENTIFICACION  DE PROYECTO'].fillna(method='ffill')

In [10]:
produccion.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 145 entries, 1 to 546
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   FECHA                          145 non-null    datetime64[ns]
 1   IDENTIFICACION  DE PROYECTO    145 non-null    object        
 2   IDENTIFICACION DE PLANO        121 non-null    object        
 3   OPERACIÓN                      143 non-null    object        
 4   DESCRIPCIÓN OPERACIÓN          145 non-null    object        
 5   CANTIDAD (UNIDADES POR PLANO)  142 non-null    float64       
 6   TIEMPO DE EJECUCIÓN REAL       145 non-null    object        
 7   Unnamed: 9                     145 non-null    object        
 8   Unnamed: 10                    145 non-null    object        
 9   TAKE TIME                      145 non-null    object        
 10  Unnamed: 12                    145 non-null    object        
 11  Unnamed: 13        

In [11]:
# Renombramos columnas 'TIEMPO DE EJECUCIÓN REAL' a 'Hora inicio' y 'Unnamed: 9' a 'Hora fin' para facilitar el análisis
produccion = produccion.rename(columns={'TIEMPO DE EJECUCIÓN REAL': 'Hora inicio', 'Unnamed: 9': 'Hora fin'})   


In [12]:
produccion.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 145 entries, 1 to 546
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   FECHA                          145 non-null    datetime64[ns]
 1   IDENTIFICACION  DE PROYECTO    145 non-null    object        
 2   IDENTIFICACION DE PLANO        121 non-null    object        
 3   OPERACIÓN                      143 non-null    object        
 4   DESCRIPCIÓN OPERACIÓN          145 non-null    object        
 5   CANTIDAD (UNIDADES POR PLANO)  142 non-null    float64       
 6   Hora inicio                    145 non-null    object        
 7   Hora fin                       145 non-null    object        
 8   Unnamed: 10                    145 non-null    object        
 9   TAKE TIME                      145 non-null    object        
 10  Unnamed: 12                    145 non-null    object        
 11  Unnamed: 13        

In [13]:
# Renombramos la columna 'Unnamed: 10' a 'Tiempo paradas', 'TAKE TIME' a 'Minutos totales de produccion', 'Unnamed: 12' a 'Unidades' y 'Unnamed: 13' a 'resultado MinxU'
produccion = produccion.rename(columns={'Unnamed: 10': 'Tiempo paradas', 'TAKE TIME': 'Minutos totales de produccion', 'Unnamed: 12': 'Unidades', 'Unnamed: 13': 'resultado MinxU'})

In [14]:
# Los campos 'LIBERACION PIEZAS (LIDER)', 'Unnamed: 16'"F-PROD-01-B registro produccion prueba.xlsx", 'Unnamed: 17' y 'Unnamed: 18' no contienen información relevante, por lo que los eliminamos utilizando el método drop() con el parámetro columns=['LIBERACION PIEZAS (LIDER)', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18']
produccion = produccion.drop(columns=['LIBERACION PIEZAS (LIDER)', 'Unnamed: 16', 'Unnamed: 17'])

In [51]:
produccion.head(20)

,FECHA,IDENTIFICACION DE PROYECTO,IDENTIFICACION DE PLANO,OPERACIÓN,DESCRIPCIÓN OPERACIÓN,CANTIDAD (UNIDADES POR PLANO),Hora inicio,Hora fin,Tiempo paradas,Minutos totales de produccion,Unidades,resultado MinxU,OBSERVACION,empleado
1,2026-03-17,G800,P01 03 009,FRESADO,CORREDERA EN 4 PERFORACIONES,1.0,07:10:00,09:43:00,10,143.0,1,143.0,NaN,ALARCON
2,2026-03-17,G800,P01 06 001,FRESADO,CORREDERA EN 8 PERFORACIONES,1.0,09:50:00,10:05:00,2,13.0,1,13.0,NaN,ALARCON
3,2026-03-17,G800,SIN DATO,ARMADORA,REFORMA PALLETIZADORA,1.0,10:07:00,11:08:00,11,50.0,1,50.0,NaN,ALARCON
4,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,CORTE CHAPA,CORTAR TAPA SOPLADOR INOX 200 x 110,2.0,11:10:00,11:25:00,2,13.0,2,6.5,EN TOTAL PARA EL TRABAJO DE LA BAÑADORA QUE FU...,ALARCON
5,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,CORTE CHAPA,CORTAR TAPA SOPLADOR INOX 200 x 110,2.0,11:25:00,12:09:00,5,39,2,19.5,NaN,ALARCON
6,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,LIMPIEZA CHAPA,LIMPIEZA DE CHAPAS CORTADAS QUITAR REBARBA,2.0,12:12:00,13:40:00,43,45,2,22.5,NaN,ALARCON
7,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,CORTE CAÑO,CORTE CAÑO TAPA SOPLADOR,2.0,13:45:00,13:55:00,5,5.0,2,2.5,NaN,ALARCON
8,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,LIMPIEZA Y PERFORADO,LIMPIEZA Y PERFORADO DE SOPORTE,2.0,13:55:00,14:40:00,0,45,2,22.5,NaN,ALARCON
9,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,PERFORADO,PERFORACION EN BAÑADORA,4.0,14:40:00,15:25:00,15,30,4,7.5,NaN,ALARCON
10,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,PERFORADO,PERFORACION EN BAÑADORA PARA SOPORTE,2.0,15:25:00,16:00:00,12,23.0,2,11.5,NaN,ALARCON


In [15]:
produccion.describe(include='all')

C:\Users\usuario\AppData\Local\Temp/ipykernel_22012/3520061925.py:1: FutureWarning: Treating datetime data as categorical rather than numeric in `.describe` is deprecated and will be removed in a future version of pandas. Specify `datetime_is_numeric=True` to silence this warning and adopt the future behavior now.
  produccion.describe(include='all')


,FECHA,IDENTIFICACION DE PROYECTO,IDENTIFICACION DE PLANO,OPERACIÓN,DESCRIPCIÓN OPERACIÓN,CANTIDAD (UNIDADES POR PLANO),Hora inicio,Hora fin,Tiempo paradas,Minutos totales de produccion,Unidades,resultado MinxU,OBSERVACION,empleado
count,145,145,121,143,145,142.000000,145,145,145.0,145.0,145.0,145.0,10,145
unique,12,14,29,60,130,NaN,60,60,23.0,111.0,9.0,115.0,10,8
top,2026-03-17 00:00:00,G800,SIN DATO,LIMPIEZA,CAMBIO DE POSICION Y RE CONEXIÓN,NaN,07:00:00,16:00:00,0.0,120.0,1.0,40.0,EN TOTAL PARA EL TRABAJO DE LA BAÑADORA QUE FU...,ALARCON
freq,40,55,40,16,4,NaN,34,31,60.0,5.0,101.0,4.0,1,35
first,2026-03-17 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last,2026-04-06 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,3.563380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,7.792035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
produccion.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 145 entries, 1 to 546
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   FECHA                          145 non-null    datetime64[ns]
 1   IDENTIFICACION  DE PROYECTO    145 non-null    object        
 2   IDENTIFICACION DE PLANO        121 non-null    object        
 3   OPERACIÓN                      143 non-null    object        
 4   DESCRIPCIÓN OPERACIÓN          145 non-null    object        
 5   CANTIDAD (UNIDADES POR PLANO)  142 non-null    float64       
 6   Hora inicio                    145 non-null    object        
 7   Hora fin                       145 non-null    object        
 8   Tiempo paradas                 145 non-null    object        
 9   Minutos totales de produccion  145 non-null    object        
 10  Unidades                       145 non-null    object        
 11  resultado MinxU    

In [77]:
# Convertimos las columnas 'Hora inicio' y 'Hora fin' a formato hora
#produccion['Hora inicio'] = pd.to_datetime(produccion['Hora inicio'], errors='coerce').dt.time
#produccion['Hora fin'] = pd.to_datetime(produccion['Hora fin'], errors='coerce').dt.time
produccion['Hora inicio'] = pd.to_datetime(produccion['Hora inicio'], format='%H:%M:%S').dt.time
produccion['Hora fin'] = pd.to_datetime(produccion['Hora fin'], format='%H:%M:%S').dt.time


In [17]:
produccion.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 145 entries, 1 to 546
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   FECHA                          145 non-null    datetime64[ns]
 1   IDENTIFICACION  DE PROYECTO    145 non-null    object        
 2   IDENTIFICACION DE PLANO        121 non-null    object        
 3   OPERACIÓN                      143 non-null    object        
 4   DESCRIPCIÓN OPERACIÓN          145 non-null    object        
 5   CANTIDAD (UNIDADES POR PLANO)  142 non-null    float64       
 6   Hora inicio                    145 non-null    object        
 7   Hora fin                       145 non-null    object        
 8   Tiempo paradas                 145 non-null    object        
 9   Minutos totales de produccion  145 non-null    object        
 10  Unidades                       145 non-null    object        
 11  resultado MinxU    

In [79]:
produccion.head()

,FECHA,IDENTIFICACION DE PROYECTO,IDENTIFICACION DE PLANO,OPERACIÓN,DESCRIPCIÓN OPERACIÓN,CANTIDAD (UNIDADES POR PLANO),Hora inicio,Hora fin,Tiempo paradas,Minutos totales de produccion,Unidades,resultado MinxU,OBSERVACION,empleado
1,2026-03-17,G800,P01 03 009,FRESADO,CORREDERA EN 4 PERFORACIONES,1.0,07:10:00,09:43:00,10,143.0,1,143.0,NaN,ALARCON
2,2026-03-17,G800,P01 06 001,FRESADO,CORREDERA EN 8 PERFORACIONES,1.0,09:50:00,10:05:00,2,13.0,1,13.0,NaN,ALARCON
3,2026-03-17,G800,SIN DATO,ARMADORA,REFORMA PALLETIZADORA,1.0,10:07:00,11:08:00,11,50.0,1,50.0,NaN,ALARCON
4,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,CORTE CHAPA,CORTAR TAPA SOPLADOR INOX 200 x 110,2.0,11:10:00,11:25:00,2,13.0,2,6.5,EN TOTAL PARA EL TRABAJO DE LA BAÑADORA QUE FU...,ALARCON
5,2026-03-17,BAÑADORA EMPANADAS,SIN PLANO,CORTE CHAPA,CORTAR TAPA SOPLADOR INOX 200 x 110,2.0,11:25:00,12:09:00,5,39,2,19.5,NaN,ALARCON


In [18]:
# Guardar el dataset limpio en un nuevo archivo Excel llamado F-PROD-01_limpio.xlsx utilizando el método to_excel() de pandas, sin incluir el índice
produccion.to_excel('F-PROD-01_limpio.xlsx', index=False)